# Forward vs Backward — Side-by-Side Comparison

Both passes share the same **X**, **W**, and **N = X @ W**.  
They diverge at **N**: the forward applies σ and sums to a scalar **L**; the backward applies σ' and multiplies by Wᵀ to get the **gradient**.

```
              X  ──────────  W
               ╲            ╱
                N  =  X @ W          ← shared
               ╱            ╲
  FORWARD                      BACKWARD
  S = σ(N)                     dS/dN = σ'(N)
  L = Σ S   (scalar)           dN/dX = Wᵀ
                               grad  = dS/dN @ Wᵀ
```

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import ipywidgets as widgets
from IPython.display import display

FWD = "#16a34a"   # green  — forward
BWD = "#dc2626"   # red    — backward
SHR = "#1e40af"   # blue   — shared

def sigmoid(x):    return 1 / (1 + np.exp(-x))
def relu(x):       return np.maximum(0, x)
def tanh_(x):      return np.tanh(x)
def leaky_relu(x): return np.where(x > 0, x, 0.01 * x)
def linear(x):     return x
SIGMAS = {"sigmoid": sigmoid, "tanh": tanh_, "relu": relu,
          "leaky_relu": leaky_relu, "linear": linear}

def deriv(func, inp, delta=0.001):
    return (func(inp + delta) - func(inp - delta)) / (2 * delta)

def heatmap(ax, data, title, title_color="black", cmap="RdBu_r", center=None):
    vmax = np.abs(data).max() or 1.0
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax) if center == 0 else None
    im = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto", interpolation="nearest")
    plt.colorbar(im, ax=ax, shrink=0.75, pad=0.02)
    r, c = data.shape
    for ri in range(r):
        for ci in range(c):
            v = data[ri, ci]
            col = "white" if abs(v) > 0.6 * vmax else "black"
            ax.text(ci, ri, f"{v:.2f}", ha="center", va="center",
                    fontsize=max(5, 9 - max(r, c)), color=col)
    ax.set_title(title, fontsize=10, fontweight="bold", color=title_color, pad=4)
    ax.set_xticks(range(c)); ax.set_yticks(range(r))
    ax.set_xticklabels([f"c{i}" for i in range(c)], fontsize=7)
    ax.set_yticklabels([f"r{i}" for i in range(r)], fontsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(title_color); spine.set_linewidth(1.6)

def section_label(ax, text, color):
    ax.axis("off")
    ax.text(0.5, 0.5, text, ha="center", va="center",
            fontsize=13, fontweight="bold", color="white",
            bbox=dict(boxstyle="round,pad=0.5", fc=color, ec=color))

print("Setup complete.")

Setup complete.


In [2]:
def draw(batch, in_feat, out_feat, sigma_name, seed):
    sigma = SIGMAS[sigma_name]
    rng   = np.random.default_rng(seed)
    X     = rng.standard_normal((batch, in_feat))
    W     = rng.standard_normal((in_feat, out_feat))

    # ── shared ────────────────────────────────────────────────────────────
    N     = X @ W

    # ── forward ───────────────────────────────────────────────────────────
    S     = sigma(N)
    L     = float(np.sum(S))
    S_flat = S.flatten()

    # ── backward ──────────────────────────────────────────────────────────
    dSdN  = deriv(sigma, N)
    dNdX  = W.T
    grad  = dSdN @ dNdX

    # ── σ curves ──────────────────────────────────────────────────────────
    N_flat = N.flatten()
    pad    = max(2.5, np.abs(N_flat).max() * 0.4)
    xs     = np.linspace(N_flat.min() - pad, N_flat.max() + pad, 500)
    ys     = sigma(xs)
    dys    = deriv(sigma, xs)

    # ══════════════════════════════════════════════════════════════════════
    # Layout:  6 cols × 6 rows
    #   col 0-2  = forward side (green border)
    #   col 3-5  = backward side (red border)
    #   row 0    = section headers
    #   row 1    = shared inputs X, W  (spans all 6 cols, 3 each)
    #   row 2    = shared N  (spans all 6 cols)
    #   row 3    = S | dSdN  dNdX
    #   row 4    = bar+L | gradient
    #   row 5    = σ curve | σ' curve
    # ══════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(18, 17))
    fig.suptitle(
        f"Forward  vs  Backward  |  σ={sigma_name}  |  "
        f"X:{X.shape}  W:{W.shape}",
        fontsize=14, fontweight="bold", y=0.995,
    )

    gs = gridspec.GridSpec(6, 6, figure=fig, hspace=0.70, wspace=0.45)

    # ── Row 0: section headers ─────────────────────────────────────────
    section_label(fig.add_subplot(gs[0, 0:3]),
                  "FORWARD PASS   X → N → S → L", FWD)
    section_label(fig.add_subplot(gs[0, 3:6]),
                  "BACKWARD PASS   X → N → dS/dN → grad", BWD)

    # ── Row 1: shared inputs ───────────────────────────────────────────
    heatmap(fig.add_subplot(gs[1, 0:3]), X,
            f"X  {X.shape}  (shared input)", SHR, center=0)
    heatmap(fig.add_subplot(gs[1, 3:6]), W,
            f"W  {W.shape}  (shared weights)", SHR, center=0)

    # ── Row 2: shared N ────────────────────────────────────────────────
    ax_N = fig.add_subplot(gs[2, 0:6])
    heatmap(ax_N, N,
            f"N = X @ W  {N.shape}  —  shared pre-activation  (used by both passes)",
            SHR, center=0)
    # vertical divider hint
    ax_N.axvline(x=N.shape[1] - 0.5, color="white", lw=0, alpha=0)  # invisible placeholder
    fig.text(0.5, ax_N.get_position().y0 - 0.005,
             "↙  forward applies σ(N)          backward applies σ'(N)  ↘",
             ha="center", va="top", fontsize=10, color="#64748b", style="italic")

    # ── Row 3: divergence ─────────────────────────────────────────────
    heatmap(fig.add_subplot(gs[3, 0:3]), S,
            f"S = σ(N)  {S.shape}", FWD)
    heatmap(fig.add_subplot(gs[3, 3:5]), dSdN,
            f"dS/dN = σ'(N)  {dSdN.shape}", BWD)
    heatmap(fig.add_subplot(gs[3, 5:6]), dNdX,
            f"dN/dX = Wᵀ  {dNdX.shape}", BWD, center=0)

    # ── Row 4: output ─────────────────────────────────────────────────
    # Forward: bar chart + scalar L
    ax_bar = fig.add_subplot(gs[4, 0:2])
    colours = [FWD if v >= 0 else "#f87171" for v in S_flat]
    labels  = [f"S[{r},{c}]" for r in range(S.shape[0]) for c in range(S.shape[1])]
    ax_bar.bar(range(len(S_flat)), S_flat, color=colours, edgecolor="white", lw=0.5)
    ax_bar.axhline(0, color="black", lw=0.7)
    ax_bar.axhline(L / len(S_flat), color="orange", lw=1.4, ls="--",
                   label=f"mean = {L/len(S_flat):.3f}")
    ax_bar.set_xticks(range(len(S_flat)))
    ax_bar.set_xticklabels(labels, rotation=55, fontsize=6.5, ha="right")
    ax_bar.set_title("S elements  (sum → L)", fontsize=10, fontweight="bold",
                     color=FWD, pad=4)
    ax_bar.legend(fontsize=8); ax_bar.grid(axis="y", alpha=0.3)
    for spine in ax_bar.spines.values():
        spine.set_edgecolor(FWD); spine.set_linewidth(1.6)

    ax_L = fig.add_subplot(gs[4, 2:3])
    ax_L.axis("off")
    ax_L.text(0.5, 0.62, "L = Σ S", ha="center", va="center",
              fontsize=11, color="gray")
    ax_L.text(0.5, 0.32, f"{L:.4f}", ha="center", va="center",
              fontsize=26, fontweight="bold", color=FWD,
              bbox=dict(boxstyle="round,pad=0.4",
                        fc="#f0fdf4", ec=FWD, lw=2))
    ax_L.set_title("Scalar loss L", fontsize=10, fontweight="bold", color=FWD)

    # Backward: gradient matrix
    heatmap(fig.add_subplot(gs[4, 3:6]), grad,
            f"GRADIENT  dS/dX = σ'(N) @ Wᵀ  {grad.shape}", BWD, center=0)

    # ── Row 5: σ and σ' curves ────────────────────────────────────────
    ax_s  = fig.add_subplot(gs[5, 0:3])
    ax_ds = fig.add_subplot(gs[5, 3:6])

    # σ(x) — forward curve
    ax_s.plot(xs, ys, color=FWD, lw=2.5, label=f"σ(x)  [{sigma_name}]")
    ax_s.scatter(N_flat, S_flat, color="#dc2626", s=50, zorder=5,
                 label=f"(N, S)  n={len(N_flat)}")
    for xv, yv in zip(N_flat, S_flat):
        ax_s.vlines(xv, 0, yv, color="#dc2626", lw=0.6, alpha=0.35, ls="--")
    ax_s.set_title(f"σ(x) — activation  (forward)", fontsize=10,
                   fontweight="bold", color=FWD)
    ax_s.set_xlabel("x  (N values)"); ax_s.set_ylabel("σ(x) = S")
    ax_s.legend(fontsize=9); ax_s.grid(True, alpha=0.3)
    for spine in ax_s.spines.values():
        spine.set_edgecolor(FWD); spine.set_linewidth(1.4)

    # σ'(x) — backward curve
    dy_pts = deriv(sigma, N_flat)
    ax_ds.plot(xs, dys, color=BWD, lw=2.5, label="σ'(x)")
    ax_ds.axhline(0, color="#9ca3af", lw=0.8, ls="--")
    ax_ds.scatter(N_flat, dy_pts, color="#f97316", s=50, zorder=5,
                  label=f"σ'(N) = dS/dN  n={len(N_flat)}")
    for xv, yv in zip(N_flat, dy_pts):
        ax_ds.vlines(xv, 0, yv, color="#f97316", lw=0.6, alpha=0.35, ls="--")
    ax_ds.set_title("σ'(x) — derivative  (backward)", fontsize=10,
                    fontweight="bold", color=BWD)
    ax_ds.set_xlabel("x  (N values)"); ax_ds.set_ylabel("σ'(x) = dS/dN")
    ax_ds.legend(fontsize=9); ax_ds.grid(True, alpha=0.3)
    for spine in ax_ds.spines.values():
        spine.set_edgecolor(BWD); spine.set_linewidth(1.4)

    # vertical dividing line between the two sides
    line = plt.Line2D([0.505, 0.505], [0.02, 0.955],
                      transform=fig.transFigure,
                      color="#cbd5e1", linewidth=1.5, linestyle="--")
    fig.add_artist(line)

    plt.show()
    print(f"Forward:  L = {L:.6f}")
    print(f"Backward: grad shape = {grad.shape},  "
          f"min={grad.min():.4f}  max={grad.max():.4f}")

print("Draw function ready.")

Draw function ready.


In [3]:
style = {"description_width": "160px"}
layout = widgets.Layout(width="440px")

w_batch    = widgets.IntSlider(value=3, min=1, max=8, description="batch (rows X)",           style=style, layout=layout)
w_in_feat  = widgets.IntSlider(value=4, min=1, max=8, description="in_feat (cols X = rows W)", style=style, layout=layout)
w_out_feat = widgets.IntSlider(value=2, min=1, max=8, description="out_feat (cols W)",         style=style, layout=layout)
w_sigma    = widgets.Dropdown(options=list(SIGMAS.keys()), value="sigmoid",
                              description="σ (activation)", style=style)
w_seed     = widgets.IntSlider(value=42, min=0, max=999, description="random seed",            style=style, layout=layout)

ui = widgets.VBox([
    widgets.HTML("<b>Shared matrix dimensions</b>"),
    w_batch, w_in_feat, w_out_feat,
    widgets.HTML("<b>Activation &amp; randomness</b>"),
    w_sigma, w_seed,
])

out = widgets.interactive_output(
    draw,
    {"batch": w_batch, "in_feat": w_in_feat, "out_feat": w_out_feat,
     "sigma_name": w_sigma, "seed": w_seed},
)

display(ui, out)

Output()